### **O — Open / Closed Principle (OCP)**
Definition: "Software entities should be **OPEN** for extension bot **CLOSED** for modification."

That means: <br>
- You should be able to add new behavior.
- **WITHOUT** changing existing, tested code.

Why OCP matters in backend systems? <br>
- Requirements change frequently
- New payment methods, discounts, AI models, rules
- Changing old code = introduces bugs

OCP helps you: <br>
- Add features safely
- Avoid regressions
- Scale teams

**BAD DESIGN (Violates OCP)** <br>

Example: Payment processing

In [35]:
class PaymentService:
    def pay(self, amount, method):
        if method == "card":
            print(f"Paid {amount} by card") 
        elif method == "paypal":
            print(f"Paid {amount} by paypal")  
        elif method == "stripe":
            print(f"Paid {amount} by Stripe") 

**Why this violates OCP** <br>

Every time you add a new payment method: <br>
- You modify PaymentService
- Risk breaking existing logic
- Tests must be re-run fully

👉 This class is OPEN for modification which is violate OCP

**GOOD DESIGN (Follows OCP)** <br>

We will use: <br>
- Strategy Pattern (behavior)
- Factory Pattern (creation)
- Adapter Pattern (3rd-party SDKs)

**Strategy Interface**

In [2]:
class PaymentStrategy:
    def pay(self, amount):
        raise NotImplementedError

**Concrete Strategies (Extensions) Open for modification**

In [11]:
class CardPayment(PaymentStrategy):
    def pay(self, amount):
        print(f"Paid ${amount} using card")


class PaypalPayment(PaymentStrategy):
    def pay(self, amount):
        print(f"Paid ${amount} using paypal")

**Adding new Payment**

In [19]:
class StripePayment(PaymentStrategy):
    def pay(self, amount):
        print(f"Paid ${amount} using stripe")

class BkashPayment(PaymentStrategy):
    def pay(self, amount):
        print(f"Paid ${amount} using bkash")

✔ No existing code modified

**Factory Pattern (to hide Object Creation Complexity) open for modification**

In [21]:
class PaymentFactory:
    _strategies = {
        "card": CardPayment,
        "paypal": PaypalPayment,
        "stripe": StripePayment,
        "bkash": BkashPayment,
    }

    @staticmethod
    def get_strategy(method):
        if method not in PaymentFactory._strategies:
            raise ValueError("Unsupported payment metod")

        return PaymentFactory._strategies[method]()

**Context (Closed for modification)**

In [25]:
class PaymentService:
    def __init__(self, strategy: PaymentStrategy):
        self.strategy = strategy
    
    def checkout(self, amount):
        self.strategy.pay(amount)

**Usage**

In [26]:
strategy = PaymentFactory.get_strategy(method="bkash")
service = PaymentService(strategy=strategy)
service.checkout(500)

Paid $500 using bkash


**Staff Engineer Insight**

Notice:
- PaymentService NEVER changes
- Adding new payments = add new class only
- Existing code remains untouched

✔ Textbook OCP

**Adapter + OCP (Real-world scenario)**

Imagine Stripe SDK:

In [27]:
class StripeSDK:
    def make_payment(self, value):
        print(f"Stripe charged ${value}") 

**Adapter**

In [28]:
class StripeAdapter(PaymentStrategy):
    def __init__(self):
        self.stripe = StripeSDK()

    def pay(self, amount):
        self.stripe.make_payment(amount)

In [ ]:
class PaymentFactory:
    _strategies = {
        "card": CardPayment,
        "paypal": PaypalPayment,
        "stripe": StripeAdapter,
        "bkash": BkashPayment,
    }

    @staticmethod
    def get_strategy(method):
        if method not in PaymentFactory._strategies:
            raise ValueError("Unsupported payment metod")

        return PaymentFactory._strategies[method]()

**Usage**

In [34]:
strategy = PaymentFactory.get_strategy(method="stripe")
service = PaymentService(strategy=strategy)
service.checkout(999)

Debug
Stripe charged $999


✔ You extend system <br>
✔ You don’t modify core logic